# AgentBlame / DCFA with Qwen2.5-14B (one model split across both GPUs)

Same experiment as the 7B run, with a stronger judge: **Qwen2.5-14B-Instruct-1M, Q4_K_M** (same publisher and quantization as the 7B).
The model plus its 128k context does not fit on one T4, so one Ollama server spreads it over **both** T4s and the
context cache is stored at 8-bit (`OLLAMA_KV_CACHE_TYPE=q8_0`) so it fits. One log is processed at a time.

**Start it with Save Version -> Save & Run All (Commit).** It stops itself after `HOURS` and saves everything to the Output tab.
Results go to `results/Qwen2.5-14B-Instruct-1M-GGUF_Q4_K_M/`, separate from the 7B results.

Before saving: Input -> your DCFA dataset (and from the 2nd run on, **this** notebook's latest Output, not the 7B one);
Accelerator -> **GPU T4 x2**; Internet -> **On**.


In [ ]:
%%bash
# SETTINGS: change only the two numbers
mkdir -p /kaggle/temp
echo 30 > /kaggle/temp/setting_SAMPLE     # logs per dataset (same logs as the 7B sample). Empty = all logs
echo 11 > /kaggle/temp/setting_HOURS      # stop after this many hours (Kaggle stops a run at 12 h; keep a margin)
date +%s > /kaggle/temp/setting_START
echo "hf.co/bartowski/Qwen2.5-14B-Instruct-1M-GGUF:Q4_K_M" > /kaggle/temp/setting_MODEL


In [ ]:
%%bash
# STEP 1: code to /kaggle/temp (Kaggle refuses '&' in file names, so Who&When is uploaded as Who_and_When);
# results, logs and cache go to /kaggle/working, which is saved as the Output
SRC=$(dirname "$(find /kaggle/input -name run.py -path '*DCFA*' -not -path '*/results*' | head -1)")
if [ "$SRC" = "." ]; then echo "ERROR: DCFA dataset not found. Add Input -> your DCFA dataset"; exit 1; fi
CODE=/kaggle/temp/DCFA
rm -rf "$CODE" && mkdir -p "$CODE" && cp -r "$SRC"/. "$CODE"/ && mv "$CODE/Who_and_When" "$CODE/Who&When"
rm -rf "$CODE/.cache"
mkdir -p /kaggle/working/{results,logs,.cache}
# Continue from a previous run of this notebook if its Output is attached as input
echo "Inputs attached:"; ls /kaggle/input
# Any depth: Kaggle mounts a notebook's output several folders down. A finished-results folder holds the model folder.
for d in $(find /kaggle/input -type d -name results -not -path '*/DCFA/*' 2>/dev/null); do
  P=$(dirname "$d"); for x in results logs .cache; do [ -d "$P/$x" ] && cp -rn "$P/$x"/. /kaggle/working/$x/; done
  echo "Restored previous results from $P"
done
for x in results logs .cache; do ln -sfn /kaggle/working/$x "$CODE/$x"; done
echo "Finished logs so far:"; cat /kaggle/working/results/*/*/*.jsonl 2>/dev/null | wc -l
# Fallback: only the zip of a previous run is visible -> unpack it (results and logs; the cache is not in the zip)
if [ -z "$(find /kaggle/input -type d -name results -not -path '*/DCFA/*' 2>/dev/null)" ]; then
  Z=$(find /kaggle/input -name all_results.zip 2>/dev/null | head -1)
  if [ -n "$Z" ]; then
    (cd /kaggle/working && unzip -oq "$Z" 'results/*' 'logs/*') && echo "Restored previous results from $Z"
    echo "Finished logs now:"; cat /kaggle/working/results/*/*/*.jsonl 2>/dev/null | wc -l
  elif [ -n "$(find /kaggle/input -name summary.txt -not -path '*/DCFA/*' 2>/dev/null)" ]; then
    echo "ERROR: a previous run's output is attached but its results were not found. What is attached:"
    find /kaggle/input -maxdepth 5 -not -path '*/DCFA/*' | head -40; exit 1
  else
    echo "(no previous results attached: starting from scratch)"
  fi
fi


In [ ]:
%%bash
# STEP 2: install Ollama, start ONE server that spreads the 14B model over both GPUs, check it really is all on the GPU
set -o pipefail
nvidia-smi --query-gpu=name,memory.total --format=csv || { echo "ERROR: no GPU. Accelerator -> GPU T4 x2"; exit 1; }
[ "$(nvidia-smi -L | wc -l)" -ge 2 ] || { echo "ERROR: the 14B model needs both T4s. Accelerator -> GPU T4 x2"; exit 1; }
curl -sI --max-time 15 https://ollama.com > /dev/null || { echo "ERROR: no internet. Settings -> Internet On"; exit 1; }
apt-get install -y -qq zstd pciutils > /dev/null 2>&1
curl -fsSL https://ollama.com/install.sh | sh > /dev/null || { echo "ERROR: Ollama install failed"; exit 1; }
# SCHED_SPREAD: split the model over both GPUs. KV q8_0 (needs flash attention): halves the 128k context cache so it fits.
CUDA_VISIBLE_DEVICES=0,1 OLLAMA_SCHED_SPREAD=1 OLLAMA_FLASH_ATTENTION=1 OLLAMA_KV_CACHE_TYPE=q8_0 \
  OLLAMA_HOST=127.0.0.1:11434 OLLAMA_NUM_PARALLEL=1 OLLAMA_KEEP_ALIVE=-1 \
  setsid nohup ollama serve >> /kaggle/working/logs/ollama_14b.log 2>&1 < /dev/null &
for i in $(seq 60); do curl -s localhost:11434/api/version > /dev/null && break; sleep 2; done
curl -s localhost:11434/api/version > /dev/null || { echo "ERROR: Ollama did not start:"; tail -20 /kaggle/working/logs/ollama_14b.log; exit 1; }
echo "Ollama is up"
cd /kaggle/temp/DCFA && set -a && source .env && set +a
MODEL=$(cat /kaggle/temp/setting_MODEL)
ollama pull "$MODEL" 2>&1 | tail -1 || exit 1
ollama pull "$OLLAMA_EMBED_MODEL" 2>&1 | tail -1 || exit 1
pip install -q python-dotenv tqdm
# Warm-up at the full 128k context, then make sure nothing spilled to the CPU (that would be many times slower)
curl -s --max-time 900 localhost:11434/api/chat -d "{\"model\":\"$MODEL\",\"messages\":[{\"role\":\"user\",\"content\":\"Say ok\"}],\"stream\":false,\"options\":{\"num_ctx\":$OLLAMA_CONTEXT_TOKENS,\"num_predict\":5}}" > /dev/null
ollama ps
nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv
if ! ollama ps | grep -F "14B" | grep -q "100% GPU"; then
  echo "ERROR: the 14B model does not fit fully on the two GPUs at 128k (see PROCESSOR above). Stopping to save quota."
  grep -iE "offload|layers|KV buffer|error" /kaggle/working/logs/ollama_14b.log | tail -15
  exit 1
fi
echo "OK: 14B model is 100% on the GPUs"


In [ ]:
%%bash
# STEP 3: the run (one worker: the model uses both GPUs)
cd /kaggle/temp/DCFA
export OLLAMA_MODEL=$(cat /kaggle/temp/setting_MODEL)   # overrides .env; results go to their own folder
SAMPLE=$(cat /kaggle/temp/setting_SAMPLE); HOURS=$(cat /kaggle/temp/setting_HOURS)
export DEADLINE=$(( $(cat /kaggle/temp/setting_START) + HOURS * 3600 - 900 ))   # stop 15 min early to save the Output
export CLAIMS=/kaggle/temp/claims PY=python SAMPLE
rm -rf "$CLAIMS"
OLLAMA_URL=http://localhost:11434 WORKER="14b " bash run_all.sh > /kaggle/working/logs/worker_14b.log 2>&1
echo "== progress"; tail -20 logs/progress.txt


In [ ]:
%%bash
# STEP 4: what is finished, speed per method, the comparison table, and one zip of everything
cd /kaggle/temp/DCFA
R=results/Qwen2.5-14B-Instruct-1M-GGUF_Q4_K_M
echo "== finished logs per dataset / method"
for f in $R/*/*.jsonl; do printf "%5d  %s\n" "$(grep -vc '"error"' "$f")" "${f#$R/}"; done
grep -h '"error"' $R/*/*.jsonl 2>/dev/null | wc -l | xargs echo "failed logs (retried next run):"
python - <<'EOF'
import glob, json, statistics as st
for m in ["all_at_once", "step_by_step", "binary_search", "dcfa"]:
    s = [json.loads(l)["seconds"] for p in glob.glob(f"results/Qwen2.5-14B-Instruct-1M-GGUF_Q4_K_M/*/{m}.jsonl") for l in open(p) if l.strip()]
    if s: print(f"{m:14} {len(s):5} logs  median {st.median(s):6.0f}s  mean {st.mean(s):6.0f}s")
EOF
python evaluate.py $R --summary > /kaggle/working/summary_14b.txt 2>&1; head -60 /kaggle/working/summary_14b.txt
cd /kaggle/working && rm -f all_results_14b.zip && zip -rq all_results_14b.zip results logs summary_14b.txt && ls -lh all_results_14b.zip
